# TP 3 : Fine-Tuning et alignement d'un LLM

## Introduction

On forme 4 équipes et on fait tourner les rôles suivants

- Défenseurs (Blue Team) doivent configurer une IA pour empêcher certains comportement
- Attaquants (Red Team) provoquer des comportements indésirables
- 2 teams évaluent les réponses et valident


Défis :

- l'IA ne doit pas répondre avec la lettre "e"
- l'IA doit aider sur des problèmes de math sans donner la réponse (utiliser MMLU)
- l'IA doit répondre seulement avec du code python valide, mais sans string ni commentaire
- l'IA ne doit répondre qu'en trois mots
- ....


## Partie 1 : Modèles commerciaux

Blue Team choisit secrètement son modèle et le configure via le premier prompt.


## Partie 2 : Avec Qwen3-0.6B configuré par système prompt

### 2.1 Installation et configuration de la version MLX de Qwen3-0.6B-Base

Tout d'abord mettez ce fichier dans votre dossier `3_ia` ou un sous-dossier
pour votre TP. Ensuite, assurez-vous que vous êtes dans un environnement
virtuel. Pour cela, référez-vous aux instructions dans `tp2.pptx`.


Pour ce TP nous allons travailler avec la version MLX de Qwen3
qui est optimisée pour les puces Apple M3. Cela va accélérer
l'entraînement du modèle.

On installe d'abord la librairie `mlx-lm`.

In [1]:
# ne faire qu'une seule fois pour tous les TP
%pip install mlx-lm 
%pip install lora

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 574.3/574.3 kB 20.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 29.3 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 14.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [mlx-lm]2m4/5 [mlx-lm]al]
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


On télécharge maintenant les fichiers du modèle `mlx-community/Qwen3-0.6B-8bit`.

1. Aller sur la page web du modèle sur [HuggingFace](https://huggingface.co)
2. Trouver l'onglet des fichiers du modèle sur HuggingFace
3. Télécharger les fichiers suivants :
    * model.safetensors
    * tokenizer_config.json
    * tokenizer.json
    * config.json
4. Avec une IA, rechercher ce que contient chaque fichier
5. Déplacez les fichiers dans un dossier `mlx-model/Qwen3-0.6B/` que vous créez dans votre dossier `3_ia`.

Vous pouvez maintenant donner un prompt au modèle et il vous répondra de la même manière qu'au TP2.

In [ ]:
from mlx_lm import generate, load
# Importe les deux fonctions principales de la bibliothèque mlx_lm
# : load pour charger un modèle, et generate pour générer du texte.


model_Qwen3, tokenizer = load("./mlx-model/Qwen3-0.6B/") 
 # Charge le modèle Qwen3 (0.6 milliards de paramètres) depuis le dossier local.
 # Retourne deux objets :
 # le modèle lui-même et son tokenizer (qui convertit le texte en tokens et inversement).

# 1. On définit le prompt avec le format ChatML
# Ce format aide le modèle à distinguer les instructions du dialogue
prompt = "hello how are you my friend ?"
#Définit la question de l'utilisateur sous forme de simple chaîne de caractères.


# Construit le prompt complet au format ChatML en utilisant une f-string.
# Ce format structure la conversation en blocs délimités par <|im_start|> et <|im_end|>,
# avec trois rôles :
# system (les instructions), user (la question), et assistant (le début de la réponse attendue).


chat_prompt = f"""
<|im_start|>system
you can only answer with the word : 
<|im_end|>

<|im_start|>user
{prompt}<|im_end|>
<|im_start|> friend
"""

# 2. On génère la réponse : Appelle la fonction de génération en lui passant le modèle,
# le tokenizer, le prompt formaté, et une limite de 150 tokens pour la réponse.
response = generate(
    model_Qwen3, 
    tokenizer, 
    prompt=chat_prompt, 
    max_tokens=150
)

print(response) # Affiche la réponse générée dans le terminal.

<|im_start|>
okay, the user just said "hello how are you my friend?" and I need to respond. Let me think about how to handle this.

First, the user is greeting me, so I should acknowledge their greeting. "Hello" is a good start. Then, they asked "how are you my friend?" which is a question about their friend. Since they mentioned being a friend, I should respond in a friendly and supportive way.

I should make sure to keep the response positive and open. Maybe say something like "Hi! How are you?" to confirm. Then, add a friendly message to encourage them. Let me check if there's any specific context I need to consider, but the user hasn't provided any additional details.


6. En utilisant une IA, commentez le bout de code plus haut pour expliquer ce que fait chaque ligne.
7. Tester différentes combinaisons de messages `user` et `system`


### 2.2 Attaque / Défense

Blue Team utilise le système prompt pour modififer le comportement de Qwen3.

## Partie 3 : Supervised Fine-Tuning (SFT) 

### 3.1 Installation de la version de base de Qwen3

1. Refaites les étapes ci-dessus pour installer la version base de Qwen3 `mlx-community/Qwen3-0.6B-Base` dans le bloc suivant.
2. Essayer différents prompt en chatML et constatez que le modèle ne sait pas répondre comme dans une conversation.

## 3.2 Supervised Fine-Tuning (SFT)

Maintenant nous allons "apprendre" au LLM de base à être un assistant.

Pour cela nous utilisons le jeu de données `angeluriot/french_instruct`
disponible sur HuggingFace.

1. Naviguez sur la page HF du dataset. Que contient le dataset ? Sous quel format ? Comment a-t-il été créé ?
2. Chargez le dataset :

In [5]:
from datasets import load_dataset
import json
import random

print("Chargement du dataset french_instruct...")
ds = load_dataset("angeluriot/french_instruct", split="train")

Chargement du dataset french_instruct...


README.md: 0.00B [00:00, ?B/s]

dataset.parquet:   0%|          | 0.00/181M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/275600 [00:00<?, ? examples/s]

3. Que fais le code ci-dessous ? Affichez le dataset créé et annotez le script avec des commentaires pour expliquer chaque étape. (Utilisez une IA.)

In [3]:
formatted_data = []

for ex in ds:
    convs = ex.get('conversation', [])
    if len(convs) >= 2:

        human_text = convs[0].get('value') or convs[0].get('text')
        ai_text = convs[1].get('value') or convs[1].get('text')
        
        if human_text and ai_text:

            if 100 < len(ai_text) < 800:
                formatted_data.append({
                    "messages": [
                        {"role": "user", "content": human_text},
                        {"role": "assistant", "content": ai_text}
                    ]
                })
    
    if len(formatted_data) >= 300:
        break

print(f"Échantillon de {len(formatted_data)} exemples prêt !")

Échantillon de 300 exemples prêt !


3. Le code ci-dessus fait le fine-tuning. Avant de le lancer, demander à une IA de commenter le code et de vous expliquer ce qui va se passer.

In [7]:
import json
import os

# On s'assure que le dossier de destination existe
data_path = "./" 
os.makedirs(data_path, exist_ok=True)

# Sauvegarde en train.jsonl
with open(os.path.join(data_path, "train.jsonl"), "w", encoding="utf-8") as f:
    for entry in formatted_data:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

# Optionnel : Créer un fichier de validation rapide (les 10 derniers exemples par ex.)
with open(os.path.join(data_path, "valid.jsonl"), "w", encoding="utf-8") as f:
    for entry in formatted_data[-10:]:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"✅ Fichiers sauvegardés dans {data_path}")
print(f"📄 train.jsonl : {len(formatted_data)} exemples")

✅ Fichiers sauvegardés dans ./
📄 train.jsonl : 300 exemples


In [9]:
from mlx_lm import lora

class TrainingArgs:
    # --- Chemins ---
    model = "./mlx-model/Qwen3-0.6B-Base"        
    data = "./"                  
    adapter_path = "blitz_adapters" 
    
    # --- Paramètres d'apprentissage ---
    iters = 150                  
    batch_size = 8               
    learning_rate = 5e-5         
    lr_schedule = None           
    grad_accumulation_steps = 1
    
    # --- Optimiseur corrigé (On reste sur le strict minimum) ---
    optimizer = "adam"
    optimizer_config = {
        "adam": {
            # On retire weight_decay car MLX Adam ne l'accepte pas toujours ainsi
            "betas": [0.9, 0.999],
            "eps": 1e-8
        }
    }
    
    # --- Configuration LoRA ---
    lora_parameters = {
        "rank": 32, 
        "alpha": 64, 
        "dropout": 0.05,
        "scale": 2.0
    }
    num_layers = 16              
    lora_layers = 16             
    fine_tune_type = "lora"      
    
    # --- Validation et Reporting ---
    val_batches = 0             
    steps_per_report = 5        
    steps_per_eval = 50          
    save_every = 100             
    
    # --- Paramètres techniques obligatoires ---
    seed = 42                    
    max_seq_length = 1024        
    test = False                 
    train = True                 
    grad_checkpoint = True       
    resume_adapter_file = None
    config = None
    report_to = None             
    project_name = "blitz-qwen"
    project_dir = "blitz-qwen-logs"

args = TrainingArgs()

lora.run(args)

Loading pretrained model
Loading datasets
Training
Trainable parameters: 1.935% (11.534M/596.050M)
Starting training..., iters: 150


ValueError: Cannot use chat template functions because tokenizer.chat_template is not set and no template argument was passed! For information about writing templates and setting the tokenizer.chat_template attribute, please see the documentation at https://huggingface.co/docs/transformers/main/en/chat_templating

In [ ]:
from mlx_lm import load, generate
from mlx_lm.sample_utils import make_sampler
import json
from pathlib import Path

# Chemin vers le fichier généré par l'entraînement
config_path = Path("blitz_adapters/adapter_config.json")

# On définit la structure exacte attendue par MLX
# Note : 'rank' et 'alpha' doivent correspondre à ce que tu as mis dans TrainingArgs
new_config = {
    "lora_parameters": {
        "rank": 32,
        "alpha": 64,
        "dropout": 0.05,
        "scale": 2.0
    },
    "num_layers": 16,
    "lora_layers": 16,
    "fine_tune_type": "lora",
    "model_type": "qwen2" # Qwen3 utilise souvent l'architecture qwen2 pour la compatibilité
}

with open(config_path, "w") as f:
    json.dump(new_config, f, indent=4)

print("Fichier adapter_config.json réinitialisé avec 'lora_parameters' !")

# On charge le modèle original MAIS on applique les nouveaux adaptateurs
model_tuned, tokenizer_tuned = load(
    "./mlx-model/Qwen3-0.6B", 
    adapter_path="blitz_adapters" 
)

# Crée un sampler plus "audacieux"
mon_sampler = make_sampler(temp=0.9, top_p=0.95) 


In [ ]:
chat_prompt = f"""<|im_start|>system
<|im_start|>user
Qu'est-ce que l'intelligence artificielle ?
<|im_end|>
<|im_start|>assistant
""" # On force le début

response = generate(
    model_tuned, 
    tokenizer_tuned, 
    prompt=chat_prompt, 
    max_tokens=2000, # Augmente un peu pour lui laisser de la place
    sampler=mon_sampler
)

response = generate(model_tuned, tokenizer_tuned, prompt=chat_prompt, max_tokens=200, sampler=mon_sampler)
print(response)